In [1]:
from z3 import *

def create_integer_array(s,name,values):
    f = Function(name, IntSort(), IntSort())
    if type(values) is list:
        for index in range(len(values)):
            val = values[index]
            if type(val) is int:
                s.add(f(index)==values[index])
            elif type(val) is tuple:
                s.add(And(val[0]<=f(index),f(index)<val[1]))
    setattr(f, "name", name)
    return f

def create_integer_array_z3(s,name,values):
    f = Array(name, IntSort(), IntSort())
    if type(values) is list:
        for index in range(len(values)):
            val = values[index]
            if type(val) is int:
                s.add(f[index]==values[index])
            elif type(val) is tuple:
                s.add(And(val[0]<=f[index],f[index]<val[1]))
    f_wrap = lambda index : f[index]
    setattr(f_wrap, "name", name)
    return f_wrap

s = Solver()
#https://interestingengineering.com/culture/solve-the-open-the-lock-puzzle-that-has-internet-puzzled
a = create_integer_array(s,"a",[6,8,2])
b = create_integer_array(s,"b",[6,1,4])
c = create_integer_array(s,"c",[2,0,6])
d = create_integer_array(s,"d",[7,3,8])
e = create_integer_array(s,"e",[3,8,0])

x = create_integer_array(s,"x",[(0,10)]*3) #or s.add(ForAll([t1],Implies(And(0<=t1,t1<3),And(0<=f(index),f(index)<10)))
#s.add(Distinct(x(0),x(1),x(2)))

#s.add(And(x(0)==0,x(1)==1,x(2)==2))

t1,t2,t3,t4 = Ints('t1 t2 t3 t4')

#https://www.pinterest.com/pin/guess-the-number--692358142703517225/


#1 SOLUTION wrong
#gives few solutions(042,062) because we listed only requirements,but first condition also implies that only one is right and in its place
# s.add(Exists([t1],And(0<=t1,t1<3,a(t1)==x(t1)))) #I have written s.add(Exists([t1],Implies(And(0<=t1,t1<3),And(a(t1)==x(t1))))) which is wrong
# s.add(Exists([t1,t2],And(0<=t1,t1<3,0<=t2,t2<3,b(t1)==x(t2),t1!=t2)))
# s.add(Exists([t1,t2,t3,t4],And(0<=t1,t1<3,0<=t2,t2<3,0<=t3,t3<3,0<=t4,t4<3,c(t1)==x(t2),t1!=t2,c(t3)==x(t4),t3!=t4,t1!=t3)))
# s.add(ForAll([t1,t2],Implies(And(0<=t1,t1<3,0<=t2,t2<3),d(t1)!=x(t2)))) # it is same as s.add(Not(Exists([t1,t2],And(0<=t1,t1<3,0<=t2,t2<3,d(t1)==x(t2)))))
# s.add(Exists([t1,t2],And(0<=t1,t1<3,0<=t2,t2<3,e(t1)==x(t2),t1!=t2)))


#2 SOLUTION we assume that all nmbers of all arrays are distinct
def correct_places_n(x,a,size):
    f = Function(x.name+a.name+"_c", IntSort(),IntSort())
    s.add(f(0)==0)
    n = Int('n')
    s.add(ForAll(n,Implies(n>0,f(n)==(f(n-1)+If(x(n-1)==a(n-1),1,0)))))
    return f(size)

def wrong_places_n(x,a,size):
    f = Function(x.name+a.name+"_w", IntSort(),IntSort())
    s.add(f(0)==0)
    n,t1 = Ints('n t1')
    s.add(ForAll(n,Implies(n>0,f(n)==(f(n-1)+If(Exists([t1],And(t1<size,x(n-1)==a(t1),t1!=(n-1))),1,0)))))
    return f(size)

#s.add(And(0==x(0),4==x(1),2==x(2))) #solution

#does not ends after 310min and even when I also add right solution
s.add(And(correct_places_n(x,a,3)==1,wrong_places_n(x,a,3)==0)) 
s.add(And(correct_places_n(x,b,3)==0,wrong_places_n(x,b,3)==1)) 
s.add(And(correct_places_n(x,c,3)==0,wrong_places_n(x,c,3)==2)) 
s.add(And(correct_places_n(x,d,3)==0,wrong_places_n(x,d,3)==0)) 
s.add(And(correct_places_n(x,e,3)==0,wrong_places_n(x,e,3)==1)) 

# s.add(And(correct_places_n(x,a,3)==1))
# s.add(And(wrong_places_n(x,b,3)==1)) 
# s.add(And(wrong_places_n(x,c,3)==2)) 
# s.add(And(wrong_places_n(x,d,3)==0)) 
# s.add(And(wrong_places_n(x,e,3)==1)) 

# s.add(And(correct_places_n(x,a,3)==1))
# s.add(Exists([t1,t2],And(0<=t1,t1<3,0<=t2,t2<3,b(t1)==x(t2),t1!=t2)))
# s.add(Exists([t1,t2,t3,t4],And(0<=t1,t1<3,0<=t2,t2<3,0<=t3,t3<3,0<=t4,t4<3,c(t1)==x(t2),t1!=t2,c(t3)==x(t4),t3!=t4,t1!=t3)))
# s.add(ForAll([t1,t2],Implies(And(0<=t1,t1<3,0<=t2,t2<3),d(t1)!=x(t2)))) # it is same as s.add(Not(Exists([t1,t2],And(0<=t1,t1<3,0<=t2,t2<3,d(t1)==x(t2)))))
# s.add(Exists([t1,t2],And(0<=t1,t1<3,0<=t2,t2<3,e(t1)==x(t2),t1!=t2)))

while True:
    st=s.check()
    if st==sat:
        m=s.model()
        print(m.eval(x(0)),m.eval(x(1)),m.eval(x(2)))
        s.add(Not(And(m.eval(x(0)==x(0)),m.eval(x(1))==x(1),m.eval(x(2))==x(2))))
    else:
        print(st)
        break

In [4]:
from z3 import *

def Min(x,y):
    return If(x < y,x,y)

def create_integer_array(s,name,values):
    f = Array(name, IntSort(), IntSort())
    if type(values) is list:
        for index in range(len(values)):
            val = values[index]
            if type(val) is int:
                s.add(f[index]==values[index])
            elif type(val) is tuple:
                s.add(And(val[0]<=f[index],f[index]<val[1]))
    return f


#3 SOLUTION arrays may be not distinct
# correct_places_n is same as in case of #2
# for wrong_places_n we need to iterate over elements of x and count appearance of each element in x and a arrays(except elements which is right and same place)
#we need to find min of number of appearanceses in x and a array
#and add 1/(count of element e in x) to global sum(because we will later iterate through same elements again) and do all this operations for every element

#to define it more easily we can define what is permutiation of array x and y(count of each element should be same in x and y)
#and then say that wrong_places_n will be equal to correct_places_n of x and y where y is permitation of a so that correct_places_n is maximum
#then we need to subscribe correct_places_n(x,a) from it to get only wrong_places_n

s = Solver()


def correct_places_n(x,a,size):
    return Sum([x[i]==a[i] for i in range(size)])

def wrong_places_n(x,a,size):
    return Sum([Min(Sum([x[i]==n for i in range(size)]),Sum([a[i]==n for i in range(size)])) for n in range(0,10)])-correct_places_n(x,a,size)



#https://interestingengineering.com/culture/solve-the-open-the-lock-puzzle-that-has-internet-puzzled
a = create_integer_array(s,"a",[6,8,2])
b = create_integer_array(s,"b",[6,1,4])
c = create_integer_array(s,"c",[2,0,6])
d = create_integer_array(s,"d",[7,3,8])
e = create_integer_array(s,"e",[3,8,0])

x = create_integer_array(s,"x",[(0,10)]*3) 
#s.add(Distinct(x[0],x[1],x[2]))

s.add(And(correct_places_n(x,a,3)==1,wrong_places_n(x,a,3)==0)) 
s.add(And(correct_places_n(x,b,3)==0,wrong_places_n(x,b,3)==1)) 
s.add(And(correct_places_n(x,c,3)==0,wrong_places_n(x,c,3)==2)) 
s.add(And(correct_places_n(x,d,3)==0,wrong_places_n(x,d,3)==0)) 
s.add(And(correct_places_n(x,e,3)==0,wrong_places_n(x,e,3)==1)) 



while True:
    st=s.check()
    if st==sat:
        m=s.model()
        print(m.eval(x[0]),m.eval(x[1]),m.eval(x[2]))
        s.add(Not(And(m.eval(x[0]==x[0]),m.eval(x[1])==x[1],m.eval(x[2])==x[2])))
    else:
        print(st)
        break

0 4 2
unsat
